In [ ]:
import sys, os, json, subprocess, concurrent.futures
from pathlib import Path
base=Path("/kaggle/input")
code=next(p for p in [base/"ricetein-code", *base.glob("*/*/ricetein-code")] if p.is_dir())
assert (code/"CODE_VERSION").read_text().strip()=="d44f65f", "Expected committed deadline experiment code"
sys.path.insert(0,str(code))
from src.kaggle_bootstrap import setup_env
python=setup_env()
gpu_query=subprocess.check_output(["nvidia-smi","--query-gpu=index","--format=csv,noheader"],text=True)
gpus=[line.strip() for line in gpu_query.splitlines() if line.strip()]
assert gpus, "No CUDA GPU allocated"
print("GPU devices:",gpus,flush=True)
subprocess.run([str(python),"-c","from src.config import load_config; from src.photo_dataset import photo_cache; photo_cache(load_config('configs/wholeimage/inceptionv3_224.yaml'),224,False,workers=2)"],env={**os.environ,"PYTHONPATH":str(code),"OMP_NUM_THREADS":"2","OPENBLAS_NUM_THREADS":"1"},check=True)
# Fixed 60 fits: no test evaluation, no reduction in the declared learning-rate search.
models=["inceptionv3","efficientnetv2s","convnext_tiny","swin_tiny"]
lrs=[3e-5,1e-4,3e-4]
tasks=[(model,lr,r) for model in models for lr in lrs for r in range(1,6)]
tasks=[t for i,t in enumerate(tasks) if i%3==1]

def worker(gpu):
    env={**os.environ,"CUDA_VISIBLE_DEVICES":gpu,"PYTHONPATH":str(code),"OMP_NUM_THREADS":"2","OPENBLAS_NUM_THREADS":"1","MKL_NUM_THREADS":"1","PYTHONUNBUFFERED":"1"}
    for index,(model,lr,r) in enumerate(tasks):
        if index%len(gpus)!=gpus.index(gpu): continue
        print("GPU",gpu,"starting",model,lr,r,flush=True)
        subprocess.run([str(python),"-u","-m","src.train_wholeimage","--config",f"configs/wholeimage/{model}_224.yaml","--lr",str(lr),"--round",str(r)],env=env,check=True)
with concurrent.futures.ThreadPoolExecutor(max_workers=len(gpus)) as pool:
    for result in pool.map(worker,gpus): pass
print("Shard validation fits complete",flush=True)
